# CMRM Rhythm Lab — *Sound of Muzak*

**Computer Music Representations and Models — Chapter 3 companion lab**

This notebook follows one rhythmic object through several representations:

\[
\text{symbolic pattern}
\rightarrow
\text{synthetic audio}
\rightarrow
\text{onset evidence}
\rightarrow
\text{autocorrelation}
\rightarrow
P(\tau,\phi)
\rightarrow
\text{time-varying periodicity}.
\]

The synthetic drum pattern is based on the symbolic example developed by **Bruno Di Giorgi** during his PhD work at Politecnico di Milano under the supervision of **Augusto Sarti**. The drum sounds used here are generated procedurally in NumPy; no commercial recording is included.

For comparison with the original performance, listen separately to [Porcupine Tree — *The Sound of Muzak*](https://youtu.be/cAtsmJMw0kU?si=ylsjelUjdi_f4_Ou).

> **How to use this notebook:** in Google Colab choose **Runtime → Run all**. Locally, open it with Jupyter and run the cells from top to bottom.

## 1. The symbolic rhythmic pattern

The starting point is not an audio recording but a symbolic drum pattern. Digits mark hits; `-` marks silence. The grid is a sixteenth-note grid (`S[16]`).

The goal is to keep the **cluster** visible: kick, snare and hi-hat are analytically separable, but the musically relevant object is their coordinated compound gesture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft
from scipy.ndimage import uniform_filter1d
from IPython.display import Audio, display

PATTERN = {
    "splash":       "S[16] 6---------------------------",
    "closed_hihat": "S[16] --5-9-5-9-5-9-5-9-5-9-5-----",
    "open_hihat":   "S[16] ------------------------9---",
    "snare":        "S[16] -----9------9------9------9-",
    "kick":         "S[16] 9--9---9--9---9--9---9--9---",
}

for name, line in PATTERN.items():
    print(f"{name:13s} {line}")

In [ ]:
def symbols(line):
    return line.split(']', 1)[1].strip().replace(' ', '')

names = list(PATTERN)
max_steps = max(len(symbols(v)) for v in PATTERN.values())
M = np.zeros((len(names), max_steps))
for r, name in enumerate(names):
    s = symbols(PATTERN[name])
    for i, ch in enumerate(s):
        if ch.isdigit():
            M[r, i] = int(ch) / 9.0

fig, ax = plt.subplots(figsize=(12, 3.8))
for r, name in enumerate(names):
    hits = np.flatnonzero(M[r] > 0)
    ax.scatter(hits, np.full_like(hits, r), s=35 + 120*M[r, hits])
ax.set_yticks(range(len(names)), names)
ax.set_xlabel('Sixteenth-note cell')
ax.set_title('Symbolic drum pattern: components of one rhythmic cluster')
ax.grid(axis='x', alpha=0.25)
plt.show()

### Listening question

Before analysing anything, look at the rows above and ask: **do these rows represent independent rhythms, or components of a larger rhythmic object?** The algebraic decomposition is useful, but it need not coincide with the perceptual or performance-level decomposition.

## 2. Generate a synthetic performance

The original research code rendered the pattern through MIDI/FluidSynth. For portability, the teaching version below synthesizes simple drum-like sounds directly. This keeps the notebook self-contained and avoids distributing any commercial audio.

In [ ]:
def hit_times(line, bpm):
    s = symbols(line)
    step = 60.0 / bpm / 4.0
    return [(i*step, int(ch)/9.0) for i, ch in enumerate(s) if ch.isdigit()]

def kick(fs):
    tt = np.arange(int(.22*fs))/fs
    phase = 2*np.pi*(75*tt - 32*tt*tt)
    return np.sin(phase)*np.exp(-tt*18)

def snare(fs, rng):
    tt = np.arange(int(.16*fs))/fs
    noise = rng.standard_normal(len(tt))*np.exp(-tt*22)
    tone = .25*np.sin(2*np.pi*190*tt)*np.exp(-tt*30)
    return .65*noise + tone

def hat(fs, rng, open_=False):
    dur = .18 if open_ else .055
    tt = np.arange(int(dur*fs))/fs
    noise = rng.standard_normal(len(tt))
    noise = np.r_[noise[0], np.diff(noise)]
    return .22*noise*np.exp(-tt*(18 if open_ else 65))

def cymbal(fs, rng):
    tt = np.arange(int(.35*fs))/fs
    n = rng.standard_normal(len(tt))
    n = np.r_[n[0], np.diff(n)]
    return .18*n*np.exp(-tt*9)

def synthesize_pattern(repeats=3, bpm=90, fs=22050, seed=0):
    rng = np.random.default_rng(seed)
    maxsteps = max(len(symbols(v)) for v in PATTERN.values())
    step = 60.0/bpm/4.0
    cycle = maxsteps*step
    x = np.zeros(int((repeats*cycle + .5)*fs))
    for r in range(repeats):
        base = r*cycle
        for name, line in PATTERN.items():
            for thit, vel in hit_times(line, bpm):
                if name == 'kick': y = kick(fs)
                elif name == 'snare': y = snare(fs, rng)
                elif name == 'closed_hihat': y = hat(fs, rng, False)
                elif name == 'open_hihat': y = hat(fs, rng, True)
                else: y = cymbal(fs, rng)
                y = y*vel
                i = int((base+thit)*fs)
                j = min(len(x), i+len(y))
                x[i:j] += y[:j-i]
    peak = np.max(np.abs(x))
    if peak > 0:
        x = .9*x/peak
    return x, fs

x, fs = synthesize_pattern()
print(f'{len(x)/fs:.2f} s at {fs} Hz')
display(Audio(x, rate=fs))

In [ ]:
t = np.arange(len(x))/fs
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, x, linewidth=.6)
ax.set_xlim(0, min(8, t[-1]))
ax.set_xlabel('Time [s]')
ax.set_ylabel('Amplitude')
ax.set_title('Synthetic waveform')
plt.show()

## 3. From audio to onset evidence

A rhythm-analysis system rarely works directly on the waveform. It first computes an **onset-detection function (ODF)**: a one-dimensional signal emphasizing moments where new acoustic events appear.

Here we use complex spectral difference, a standard way to combine spectral magnitude and phase change.

In [ ]:
def complex_spectral_difference(x, fs, frame_length=1024, hop_length=256):
    _, _, Z = stft(x, fs=fs, window='hamming', nperseg=frame_length,
                   noverlap=frame_length-hop_length, nfft=frame_length,
                   boundary=None, padded=False)
    if Z.shape[1] < 3:
        return np.zeros(0)
    amp_pred = np.abs(Z[:,1:-1])
    amp_true = np.abs(Z[:,2:])
    phase = np.unwrap(np.angle(Z), axis=1)
    shift = phase[:,2:] - 2*phase[:,1:-1] + phase[:,:-2]
    shift = (shift + np.pi) % (2*np.pi) - np.pi
    d2 = amp_pred**2 + amp_true**2 - 2*amp_pred*amp_true*np.cos(shift)
    return np.sum(np.sqrt(np.maximum(d2, 0)), axis=0)

hop = 256
odf = complex_spectral_difference(x, fs, 1024, hop)
odf_rate = fs/hop
t_odf = np.arange(len(odf))/odf_rate

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_odf, odf)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Onset evidence')
ax.set_title('Complex spectral-difference onset function')
plt.show()

## 4. Ordinary autocorrelation: periodicity without phase

For a one-dimensional onset sequence \(x[n]\), ordinary autocorrelation compares the sequence with delayed copies of itself. Peaks reveal delays at which the event pattern tends to recur.

This is useful for estimating periodicity, but it largely suppresses **where within the cycle** events occur.

In [ ]:
def unbiased_autocorrelation(x, max_lag=None, normalize=True):
    x = np.asarray(x, float).reshape(-1)
    n = len(x)
    if max_lag is None: max_lag = n-1
    max_lag = min(int(max_lag), n-1)
    out = np.empty(max_lag+1)
    for lag in range(max_lag+1):
        out[lag] = np.dot(x[:n-lag], x[lag:]) / max(1, n-lag)
    if normalize and np.max(np.abs(out)) > 0:
        out /= np.max(np.abs(out))
    return out

odf0 = np.maximum(odf - uniform_filter1d(odf, size=15, mode='nearest'), 0)
max_lag = int(2.5*odf_rate)
ac = unbiased_autocorrelation(odf0, max_lag=max_lag)
lags_s = np.arange(len(ac))/odf_rate

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(lags_s[1:], ac[1:])
ax.set_xlabel('Lag [s]')
ax.set_ylabel('Normalized autocorrelation')
ax.set_title('Global periodicity evidence')
plt.show()

## 5. Phase autocorrelation \(P(\tau,\phi)\)

Bruno Di Giorgi's phase-autocorrelation representation retains both **period** and **phase**. For lag \(\tau\) and phase \(\phi<\tau\),

\[
P(\tau,\phi)=\sum_i x(\tau i+\phi)\,x(\tau(i+1)+\phi).
\]

Ordinary autocorrelation asks *“how periodic is this lag?”* Phase autocorrelation additionally asks *“at which phase does that periodicity occur?”*

In [ ]:
def phase_autocorrelation(x, lags, unbiased=True):
    x = np.asarray(x, float).reshape(-1)
    lags = np.asarray(lags, int)
    maxlag = int(lags.max())
    P = np.zeros((len(lags), maxlag))
    C = np.zeros_like(P)
    N = len(x)
    for r, tau in enumerate(lags):
        for phi in range(tau):
            idx = np.arange(phi, N, tau, dtype=int)
            if len(idx) > 1:
                P[r, phi] = np.dot(x[idx[:-1]], x[idx[1:]])
                C[r, phi] = len(idx)-1
    if unbiased:
        nz = C > 0
        P[nz] /= C[nz]
    return P

lags = np.arange(max(2, int(.20*odf_rate)), int(1.6*odf_rate))
P = phase_autocorrelation(odf0, lags)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(P, origin='lower', aspect='auto', interpolation='nearest',
               extent=[0, P.shape[1]/odf_rate, lags[0]/odf_rate, lags[-1]/odf_rate])
ax.set_xlabel('Phase $\\phi$ [s]')
ax.set_ylabel('Lag $\\tau$ [s]')
ax.set_title('Phase autocorrelation $P(\\tau,\\phi)$')
fig.colorbar(im, ax=ax, label='Correlation')
plt.show()

### What to look for

A strong ordinary-autocorrelation peak says that a period is present. A compact ridge or peak in the phase-autocorrelation plane says something richer: a periodic component is present **and occupies a particular phase relationship**. This is precisely the kind of information that matters when several rhythmic components form a coordinated cluster.

## 6. A time-varying rhythmogram

Real music changes. Instead of analysing the whole ODF at once, we can analyse overlapping windows and obtain

\[
R(t,\tau),
\]

a time-varying map of periodicity salience.

In [ ]:
def frame_signal(x, frame_length, hop_length, center=True):
    x = np.asarray(x, float).reshape(-1)
    if center:
        p = frame_length//2
        x = np.pad(x, (p,p))
    if len(x) < frame_length:
        x = np.pad(x, (0, frame_length-len(x)))
    n = 1 + (len(x)-frame_length)//hop_length
    return np.stack([x[i*hop_length:i*hop_length+frame_length] for i in range(n)], axis=1)

def rhythmogram(odf, frame_length=256, hop_length=32, period_range=(12,120)):
    F = frame_signal(odf, frame_length, hop_length, center=True)
    periods = np.arange(period_range[0], period_range[1]+1)
    R = np.zeros((len(periods), F.shape[1]))
    for j, f in enumerate(F.T):
        residual = np.maximum(f - uniform_filter1d(f, size=15, mode='nearest'), 0)
        acf = unbiased_autocorrelation(residual, max_lag=periods[-1], normalize=True)
        R[:,j] = acf[periods]
    return R, periods

R, periods = rhythmogram(odf0)
rhop = 32
t_r = np.arange(R.shape[1])*rhop/odf_rate

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(R, origin='lower', aspect='auto', interpolation='nearest',
               extent=[t_r[0], t_r[-1] if len(t_r)>1 else 1,
                       periods[0]/odf_rate, periods[-1]/odf_rate])
ax.set_xlabel('Analysis time [s]')
ax.set_ylabel('Candidate period [s]')
ax.set_title('Rhythmogram $R(t,\\tau)$')
fig.colorbar(im, ax=ax, label='Periodicity salience')
plt.show()

## 7. Relating signal analysis back to the algebra

Earlier in Chapter 3, *Sound of Muzak* is used as an example of a **systemic \(7{:}4\) rhythmic construction**, with one component translated relative to the reference frame. The algebraic description tells us what structural relations were used to construct the pattern.

The signal-analysis representations above answer a different question: **what traces of those relations survive after the symbolic pattern has become sound?**

This is the conceptual loop:

\[
\boxed{\text{sets/cosets and cluster}}
\rightarrow
\boxed{\text{synthetic performance}}
\rightarrow
\boxed{\text{onset evidence}}
\rightarrow
\boxed{\text{period + phase evidence}}.
\]

Neither representation replaces the other. One is constructive; the other is inferential.

## 8. Small experiments

Try changing one thing at a time and rerunning the notebook:

1. Change `bpm=90` in `synthesize_pattern` and verify which displays change geometrically and which remain structurally similar.
2. Mute the snare line by replacing its digits with `-`. What disappears from the onset function and the phase-autocorrelation map?
3. Shift the snare pattern by one or more sixteenth-note cells. Which representation makes the phase change easiest to see?
4. Keep the snare displacement fixed but simplify the hi-hat. Does the **cluster** remain perceptually recognizable?
5. Increase `repeats`. Does a longer observation window sharpen periodicity estimates?

The important question is not simply whether each component can be recovered independently, but whether the representations help explain why several components can behave as a **coherent rhythmic object**.

---
### Provenance and distribution

This student lab is a modern teaching reconstruction based on rhythm-analysis applications and experiments originally developed by **Bruno Di Giorgi** during his PhD work at Politecnico di Milano under the supervision of **Augusto Sarti**. The portable synthesis and notebook organization are new teaching infrastructure. Public redistribution should be accompanied by Bruno Di Giorgi's permission and an explicit software license.